In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 2: Cohort Retention Analysis
==================================================================
Purpose: Analyze user retention through cohort analysis to identify
patterns in user behavior, churn drivers, and retention optimization
opportunities.

Key Questions:
1. What is the 30-day, 60-day, 90-day retention rate by signup month?
2. Which acquisition channels have the best retention?
3. Do premium members retain better than free users?
4. How does retention vary by city and user segment?
5. What is the "repeat purchase rate" and how is it trending?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE COHORT RETENTION ANALYSIS")
print("="*80)

---------------------------------------------------------------------
1. LOAD CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading cleaned data...")

In [ ]:
# Load cleaned data from Notebook 1
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
cities = pd.read_csv('../data/cities.csv')

In [ ]:
# Convert dates
users['signup_date'] = pd.to_datetime(users['signup_date'])
users['signup_month'] = users['signup_date'].dt.to_period('M')
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])
orders['order_month'] = orders['order_placed_at'].dt.to_period('M')

In [ ]:
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(orders):,} orders")

In [ ]:
# Filter only delivered orders for retention analysis
delivered_orders = orders[orders['order_status'] == 'delivered']

In [ ]:
print(f"✅ Using {len(delivered_orders):,} delivered orders")

---------------------------------------------------------------------
2. FIRST ORDER ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FIRST ORDER ANALYSIS")
print("="*80)

In [ ]:
# Get first order for each user
first_orders = delivered_orders.sort_values('order_placed_at').groupby('user_id').first().reset_index()
first_orders['cohort_month'] = first_orders['order_placed_at'].dt.to_period('M')

In [ ]:
print(f"📊 Users with at least one delivered order: {len(first_orders):,}")
print(f"📊 Users who never ordered: {len(users) - len(first_orders):,}")

In [ ]:
# Merge with user attributes
first_orders = first_orders.merge(
    users[['user_id', 'acquisition_channel', 'is_premium_member', 'city_id', 'age_band', 'device_type']],
    on='user_id', how='left'
)

In [ ]:
# Add city names
first_orders = first_orders.merge(cities[['city_id', 'city_name', 'tier']], on='city_id', how='left')

In [ ]:
print("\n📊 First order statistics:")
print(f"  • Average first order value: ₹{first_orders['total_amount'].mean():.2f}")
print(f"  • Median first order value: ₹{first_orders['total_amount'].median():.2f}")
print(f"  • First orders by channel:")
channel_counts = first_orders['acquisition_channel'].value_counts()
for channel, count in channel_counts.items():
    print(f"    - {channel}: {count:,} ({count/len(first_orders)*100:.1f}%)")

---------------------------------------------------------------------
3. COHORT RETENTION MATRIX
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("COHORT RETENTION MATRIX")
print("="*80)

In [ ]:
def create_cohort_retention(orders_df, users_df, cohort_col='signup_month', 
                           activity_col='order_month', retention_type='user_based'):
    """
    Create a cohort retention matrix.
    
    Parameters:
    - orders_df: DataFrame with orders data
    - users_df: DataFrame with user data
    - cohort_col: Column to define cohorts (e.g., signup_month)
    - activity_col: Column to define activity periods (e.g., order_month)
    - retention_type: 'user_based' or 'order_based'
    
    Returns:
    - cohort_data: DataFrame with cohort retention matrix
    """
    
    # Get user cohorts
    user_cohorts = users_df[[cohort_col]].copy()
    user_cohorts['user_id'] = users_df['user_id']
    
    # Get user activity by period
    user_activity = orders_df.groupby(['user_id', activity_col]).size().reset_index(name='orders')
    user_activity = user_activity.merge(user_cohorts, on='user_id')
    
    # Create cohort size (unique users per cohort)
    cohort_size = user_cohorts.groupby(cohort_col).size().reset_index(name='cohort_size')
    
    # Calculate retention
    retention_data = []
    
    for cohort in user_activity[cohort_col].unique():
        cohort_users = user_cohorts[user_cohorts[cohort_col] == cohort]['user_id'].unique()
        total_users = len(cohort_users)
        
        # For each period after cohort start
        cohort_activity = user_activity[user_activity[cohort_col] == cohort]
        
        # Get all periods from cohort start to end
        all_periods = sorted(user_activity[activity_col].unique())
        cohort_start_idx = all_periods.tolist().index(cohort) if cohort in all_periods else -1
        
        if cohort_start_idx != -1:
            for period_idx, period in enumerate(all_periods[cohort_start_idx:], 0):
                active_users = cohort_activity[cohort_activity[activity_col] == period]['user_id'].nunique()
                retention_rate = active_users / total_users if total_users > 0 else 0
                
                retention_data.append({
                    'cohort': cohort,
                    'period_number': period_idx,
                    'active_users': active_users,
                    'total_users': total_users,
                    'retention_rate': retention_rate,
                    'period': period
                })
    
    retention_df = pd.DataFrame(retention_data)
    
    # Pivot to matrix format
    retention_matrix = retention_df.pivot_table(
        index='cohort', 
        columns='period_number', 
        values='retention_rate'
    )
    
    # Add cohort size as first column
    cohort_size_map = cohort_size.set_index(cohort_col)['cohort_size']
    retention_matrix['cohort_size'] = retention_matrix.index.map(cohort_size_map)
    
    return retention_matrix, retention_df

In [ ]:
# Create cohort retention matrix
print("\n🔄 Building cohort retention matrix...")

In [ ]:
# Use signup month as cohort, order month as activity
retention_matrix, retention_df = create_cohort_retention(
    delivered_orders, 
    users, 
    cohort_col='signup_month',
    activity_col='order_month'
)

In [ ]:
print(f"✅ Retention matrix shape: {retention_matrix.shape}")
print(f"✅ Cohorts: {len(retention_matrix)} months")
print(f"✅ Periods: {max(retention_matrix.columns[retention_matrix.columns != 'cohort_size'], default=0)} months")

In [ ]:
# Display the first few rows of the retention matrix
print("\n📊 Retention Matrix (first 6 cohorts):")
print(retention_matrix.head(6).round(3))

---------------------------------------------------------------------
4. VISUALIZE COHORT RETENTION HEATMAP
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("VISUALIZING COHORT RETENTION")
print("="*80)

In [ ]:
# Create figure
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('QuickBite Cohort Retention Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 4.1 Retention Heatmap
ax = axes[0, 0]
# Remove cohort_size column for heatmap
retention_heatmap = retention_matrix.drop('cohort_size', axis=1)
sns.heatmap(retention_heatmap, ax=ax, annot=True, fmt='.0%', cmap='RdYlGn', 
            cbar_kws={'label': 'Retention Rate'}, linewidths=0.5, vmin=0, vmax=1)
ax.set_title('Monthly Cohort Retention Heatmap')
ax.set_xlabel('Months Since Signup')
ax.set_ylabel('Signup Month')

In [ ]:
# 4.2 Average Retention Curve
ax = axes[0, 1]
avg_retention = retention_df.groupby('period_number')['retention_rate'].mean()
ax.plot(avg_retention.index, avg_retention.values, marker='o', linewidth=2, color='#2ecc71')
ax.fill_between(avg_retention.index, 
                avg_retention.values - avg_retention.std() / 2,
                avg_retention.values + avg_retention.std() / 2,
                alpha=0.2, color='#2ecc71')
ax.set_title('Average Retention by Period (Across All Cohorts)')
ax.set_xlabel('Months Since Signup')
ax.set_ylabel('Retention Rate')
ax.set_xticks(range(0, len(avg_retention), 2))
ax.grid(True, alpha=0.3)

In [ ]:
# Add data labels
for i, v in enumerate(avg_retention.values[:12]):
    ax.text(i, v + 0.02, f'{v:.0%}', ha='center', fontsize=8)

In [ ]:
# 4.3 Cohort-Specific Retention Curves (select top 5 cohorts)
ax = axes[1, 0]
recent_cohorts = retention_matrix.index[-min(5, len(retention_matrix)):]

In [ ]:
for cohort in recent_cohorts:
    cohort_data = retention_matrix.loc[cohort].drop('cohort_size')
    ax.plot(cohort_data.index, cohort_data.values, marker='o', 
            label=f'{cohort}', linewidth=1.5)

In [ ]:
ax.set_title('Retention Curves by Cohort (Recent Months)')
ax.set_xlabel('Months Since Signup')
ax.set_ylabel('Retention Rate')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 0.5)

In [ ]:
# 4.4 Cohort Size Distribution
ax = axes[1, 1]
cohort_sizes = retention_matrix['cohort_size'].sort_index()
ax.bar(cohort_sizes.index.astype(str), cohort_sizes.values, color='#3498db', alpha=0.7)
ax.set_title('Cohort Sizes Over Time')
ax.set_xlabel('Signup Month')
ax.set_ylabel('Number of Users')
ax.tick_params(axis='x', rotation=45)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/cohort_retention_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
5. RETENTION BY ACQUISITION CHANNEL
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("RETENTION BY ACQUISITION CHANNEL")
print("="*80)

In [ ]:
def calculate_channel_retention(orders_df, users_df, months=6):
    """Calculate retention by acquisition channel for N months"""
    
    # Get first order for each user
    first_orders = orders_df.sort_values('order_placed_at').groupby('user_id').first().reset_index()
    first_orders['first_order_month'] = first_orders['order_placed_at'].dt.to_period('M')
    
    # Merge with user channels
    first_orders = first_orders.merge(
        users_df[['user_id', 'acquisition_channel']], on='user_id', how='left'
    )
    
    # For each channel, calculate retention
    channel_retention = {}
    
    for channel in first_orders['acquisition_channel'].unique():
        channel_users = first_orders[first_orders['acquisition_channel'] == channel]['user_id'].unique()
        channel_first_orders = first_orders[first_orders['user_id'].isin(channel_users)]
        
        # Calculate orders after first order
        retention_rates = []
        
        for month in range(1, months + 1):
            # Users who ordered again within N months of first order
            active_users = 0
            total_active = len(channel_users)
            
            for user_id in channel_users:
                user_orders = orders_df[orders_df['user_id'] == user_id]
                first_order_date = channel_first_orders[channel_first_orders['user_id'] == user_id]['order_placed_at'].iloc[0]
                
                # Check if user ordered again within N months
                has_order = len(user_orders[
                    (user_orders['order_placed_at'] > first_order_date) &
                    (user_orders['order_placed_at'] <= first_order_date + timedelta(days=month*30))
                ]) > 0
                
                if has_order:
                    active_users += 1
            
            retention_rate = active_users / total_active if total_active > 0 else 0
            retention_rates.append(retention_rate)
        
        channel_retention[channel] = retention_rates
    
    return pd.DataFrame(channel_retention)

In [ ]:
print("\n🔄 Calculating channel retention...")
channel_retention = calculate_channel_retention(delivered_orders, users, months=12)

In [ ]:
# Display results
print("\n📊 Channel Retention Rates (6 months):")
channel_6month = channel_retention.iloc[5].sort_values(ascending=False)
for channel, rate in channel_6month.items():
    print(f"  • {channel}: {rate*100:.1f}%")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

In [ ]:
for channel in channel_retention.columns:
    ax.plot(channel_retention.index + 1, channel_retention[channel] * 100, 
            marker='o', label=channel, linewidth=2)

In [ ]:
ax.set_title('Retention by Acquisition Channel', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Order')
ax.set_ylabel('Retention Rate (%)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 80)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/channel_retention.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
6. PREMIUM VS NON-PREMIUM RETENTION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("PREMIUM VS NON-PREMIUM RETENTION")
print("="*80)

In [ ]:
def calculate_segment_retention(orders_df, users_df, segment_col, months=12):
    """Calculate retention by user segment"""
    
    first_orders = orders_df.sort_values('order_placed_at').groupby('user_id').first().reset_index()
    first_orders = first_orders.merge(
        users_df[['user_id', segment_col]], on='user_id', how='left'
    )
    
    segment_retention = {}
    
    for segment in first_orders[segment_col].unique():
        if pd.isna(segment):
            continue
            
        segment_users = first_orders[first_orders[segment_col] == segment]['user_id'].unique()
        
        retention_rates = []
        
        for month in range(1, months + 1):
            active_users = 0
            total_active = len(segment_users)
            
            for user_id in segment_users:
                user_orders = orders_df[orders_df['user_id'] == user_id]
                first_order_date = first_orders[first_orders['user_id'] == user_id]['order_placed_at'].iloc[0]
                
                has_order = len(user_orders[
                    (user_orders['order_placed_at'] > first_order_date) &
                    (user_orders['order_placed_at'] <= first_order_date + timedelta(days=month*30))
                ]) > 0
                
                if has_order:
                    active_users += 1
            
            retention_rate = active_users / total_active if total_active > 0 else 0
            retention_rates.append(retention_rate)
        
        segment_retention[segment] = retention_rates
    
    return pd.DataFrame(segment_retention)

In [ ]:
# Premium vs Non-premium
premium_retention = calculate_segment_retention(
    delivered_orders, users, 'is_premium_member', months=12
)

In [ ]:
print("\n📊 Premium vs Non-Premium Retention (3 months):")
print(f"  • Premium: {premium_retention.iloc[2].get(True, 0)*100:.1f}%")
print(f"  • Non-Premium: {premium_retention.iloc[2].get(False, 0)*100:.1f}%")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

In [ ]:
premium_retention.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Premium vs Non-Premium User Retention', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Order')
ax.set_ylabel('Retention Rate (%)')
ax.legend(['Premium', 'Non-Premium'])
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 60)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/premium_retention.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical test for retention difference
premium_3m = premium_retention.iloc[2]
premium_rate = premium_3m.get(True, 0)
non_premium_rate = premium_3m.get(False, 0)

In [ ]:
# Simple z-test for proportions
import math
n_premium = len(users[users['is_premium_member'] == True])
n_non_premium = len(users[users['is_premium_member'] == False])
p_pooled = (premium_rate * n_premium + non_premium_rate * n_non_premium) / (n_premium + n_non_premium)
se = math.sqrt(p_pooled * (1 - p_pooled) * (1/n_premium + 1/n_non_premium))
z_score = (premium_rate - non_premium_rate) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

In [ ]:
print(f"\n📊 Statistical Test for Premium Retention Difference:")
print(f"  • Premium 3-month retention: {premium_rate*100:.1f}%")
print(f"  • Non-Premium 3-month retention: {non_premium_rate*100:.1f}%")
print(f"  • Z-score: {z_score:.2f}")
print(f"  • P-value: {p_value:.4f}")
print(f"  • Significant at 95% confidence: {'✅ Yes' if p_value < 0.05 else '❌ No'}")

---------------------------------------------------------------------
7. CITY-LEVEL RETENTION ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CITY-LEVEL RETENTION ANALYSIS")
print("="*80)

In [ ]:
# Calculate retention by city (simplified - 3-month retention)
city_first_orders = first_orders.merge(cities[['city_id', 'city_name']], on='city_id', how='left')
city_retention = {}

In [ ]:
for city in city_first_orders['city_name'].unique():
    city_users = city_first_orders[city_first_orders['city_name'] == city]['user_id'].unique()
    
    active_at_3months = 0
    for user_id in city_users:
        user_orders = delivered_orders[delivered_orders['user_id'] == user_id]
        first_order_date = first_orders[first_orders['user_id'] == user_id]['order_placed_at'].iloc[0]
        
        has_order = len(user_orders[
            (user_orders['order_placed_at'] > first_order_date) &
            (user_orders['order_placed_at'] <= first_order_date + timedelta(days=90))
        ]) > 0
        
        if has_order:
            active_at_3months += 1
    
    retention_rate = active_at_3months / len(city_users) if len(city_users) > 0 else 0
    city_retention[city] = {
        'users': len(city_users),
        'retention_3m': retention_rate
    }

In [ ]:
city_retention_df = pd.DataFrame(city_retention).T.sort_values('retention_3m', ascending=False)

In [ ]:
print("\n📊 Top 5 Cities by 3-Month Retention:")
for city, row in city_retention_df.head(5).iterrows():
    print(f"  • {city}: {row['retention_3m']*100:.1f}% ({row['users']:,} users)")

In [ ]:
print("\n📊 Bottom 5 Cities by 3-Month Retention:")
for city, row in city_retention_df.tail(5).iterrows():
    print(f"  • {city}: {row['retention_3m']*100:.1f}% ({row['users']:,} users)")

In [ ]:
# Visualize city retention
fig, ax = plt.subplots(figsize=(14, 8))
city_retention_sorted = city_retention_df.sort_values('retention_3m', ascending=True)

In [ ]:
colors = ['#e74c3c' if x < 0.15 else '#f39c12' if x < 0.20 else '#2ecc71' 
          for x in city_retention_sorted['retention_3m']]

In [ ]:
bars = ax.barh(city_retention_sorted.index, city_retention_sorted['retention_3m'] * 100, 
               color=colors, alpha=0.7)
ax.set_title('3-Month Retention Rate by City', fontsize=14, fontweight='bold')
ax.set_xlabel('Retention Rate (%)')
ax.set_ylabel('City')
ax.set_xlim(0, 40)
ax.grid(True, alpha=0.3)

In [ ]:
# Add data labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{width:.1f}%', ha='left', va='center', fontsize=10)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/city_retention.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
8. REPEAT PURCHASE RATE ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("REPEAT PURCHASE RATE ANALYSIS")
print("="*80)

In [ ]:
def calculate_repeat_purchase_rate(orders_df, days_window=60, min_orders=2):
    """Calculate repeat purchase rate within a given window"""
    
    # Get first order for each user
    first_orders = orders_df.sort_values('order_placed_at').groupby('user_id').first().reset_index()
    
    repeat_users = 0
    for _, row in first_orders.iterrows():
        user_id = row['user_id']
        first_order_date = row['order_placed_at']
        
        # Check if user ordered again within window
        user_orders = orders_df[
            (orders_df['user_id'] == user_id) &
            (orders_df['order_placed_at'] > first_order_date) &
            (orders_df['order_placed_at'] <= first_order_date + timedelta(days=days_window))
        ]
        
        if len(user_orders) >= min_orders - 1:
            repeat_users += 1
    
    repeat_rate = repeat_users / len(first_orders) if len(first_orders) > 0 else 0
    return repeat_rate, repeat_users, len(first_orders)

In [ ]:
# Calculate repeat purchase rates for different windows
windows = [30, 60, 90, 180, 365]
repeat_rates = {}

In [ ]:
for window in windows:
    rate, repeat_users, total_users = calculate_repeat_purchase_rate(delivered_orders, window)
    repeat_rates[window] = {
        'rate': rate,
        'repeat_users': repeat_users,
        'total_users': total_users
    }
    print(f"  • {window}-day window: {rate*100:.1f}% ({repeat_users:,}/{total_users:,})")

In [ ]:
# Visualize repeat purchase rates
fig, ax = plt.subplots(figsize=(10, 6))

In [ ]:
windows_labels = [f"{w} days" for w in windows]
rates = [repeat_rates[w]['rate'] * 100 for w in windows]

In [ ]:
bars = ax.bar(windows_labels, rates, color='#3498db', alpha=0.7)
ax.set_title('Repeat Purchase Rate by Time Window', fontsize=14, fontweight='bold')
ax.set_xlabel('Window Since First Order')
ax.set_ylabel('Repeat Purchase Rate (%)')
ax.set_ylim(0, max(rates) * 1.2)

In [ ]:
for bar, rate in zip(bars, rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=10)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/repeat_purchase_rate.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
9. CHURN ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CHURN ANALYSIS")
print("="*80)

In [ ]:
# Define churn as no order in trailing 90 days
today = pd.Timestamp.now()
last_order_date = delivered_orders.groupby('user_id')['order_placed_at'].max().reset_index()
last_order_date['days_since_last_order'] = (today - last_order_date['order_placed_at']).dt.days
last_order_date['is_churned'] = last_order_date['days_since_last_order'] > 90

In [ ]:
# Merge with user attributes
churn_analysis = last_order_date.merge(
    users[['user_id', 'acquisition_channel', 'is_premium_member', 'city_id', 'age_band']],
    on='user_id', how='left'
)

In [ ]:
# Overall churn rate
churn_rate = churn_analysis['is_churned'].mean()
print(f"📊 Overall Churn Rate: {churn_rate*100:.1f}%")

In [ ]:
# Churn by acquisition channel
channel_churn = churn_analysis.groupby('acquisition_channel')['is_churned'].agg(['mean', 'count'])
channel_churn = channel_churn.sort_values('mean', ascending=False)
print("\n📊 Churn Rate by Acquisition Channel:")
for channel, row in channel_churn.iterrows():
    print(f"  • {channel}: {row['mean']*100:.1f}% ({row['count']:,} users)")

In [ ]:
# Churn by premium status
premium_churn = churn_analysis.groupby('is_premium_member')['is_churned'].mean()
print(f"\n📊 Churn Rate by Premium Status:")
print(f"  • Premium: {premium_churn.get(True, 0)*100:.1f}%")
print(f"  • Non-Premium: {premium_churn.get(False, 0)*100:.1f}%")

In [ ]:
# Visualize churn analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Churn Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 9.1 Churn by Channel
ax = axes[0, 0]
channel_churn_sorted = channel_churn.sort_values('mean', ascending=False)
colors = ['#e74c3c' if x > 0.5 else '#f39c12' if x > 0.3 else '#2ecc71' 
          for x in channel_churn_sorted['mean']]
bars = ax.bar(channel_churn_sorted.index, channel_churn_sorted['mean'] * 100, color=colors)
ax.set_title('Churn Rate by Acquisition Channel')
ax.set_ylabel('Churn Rate (%)')
ax.tick_params(axis='x', rotation=45)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

In [ ]:
# 9.2 Churn by City
ax = axes[0, 1]
city_churn = churn_analysis.merge(cities[['city_id', 'city_name']], on='city_id', how='left')
city_churn = city_churn.groupby('city_name')['is_churned'].mean().sort_values(ascending=False)
city_churn.head(10).plot(kind='barh', ax=ax, color='#e74c3c', alpha=0.7)
ax.set_title('Top 10 Cities by Churn Rate')
ax.set_xlabel('Churn Rate (%)')

In [ ]:
# 9.3 Churn by Age Band
ax = axes[1, 0]
age_churn = churn_analysis.groupby('age_band')['is_churned'].mean()
age_churn.plot(kind='bar', ax=ax, color='#3498db', alpha=0.7)
ax.set_title('Churn Rate by Age Band')
ax.set_xlabel('Age Band')
ax.set_ylabel('Churn Rate (%)')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(age_churn.values):
    ax.text(i, v + 0.01, f'{v*100:.1f}%', ha='center', va='bottom')

In [ ]:
# 9.4 Days Since Last Order Distribution
ax = axes[1, 1]
churn_analysis['days_since_last_order'].hist(bins=50, ax=ax, color='#9b59b6', alpha=0.7)
ax.axvline(90, color='red', linestyle='--', label='Churn Threshold (90 days)')
ax.set_title('Days Since Last Order Distribution')
ax.set_xlabel('Days Since Last Order')
ax.set_ylabel('Number of Users')
ax.legend()

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/churn_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
10. RETENTION DRIVERS ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("RETENTION DRIVERS ANALYSIS")
print("="*80)

Analyze what drives higher retention

In [ ]:
# Create features for retention analysis
retention_features = first_orders.merge(
    users[['user_id', 'acquisition_channel', 'is_premium_member', 'age_band', 'device_type']],
    on='user_id'
)

In [ ]:
# Create retention label (3-month retention)
def is_retained_3m(user_id):
    first_order_date = first_orders[first_orders['user_id'] == user_id]['order_placed_at'].iloc[0]
    user_orders = delivered_orders[
        (delivered_orders['user_id'] == user_id) &
        (delivered_orders['order_placed_at'] > first_order_date) &
        (delivered_orders['order_placed_at'] <= first_order_date + timedelta(days=90))
    ]
    return len(user_orders) > 0

In [ ]:
retention_features['retained_3m'] = retention_features['user_id'].apply(is_retained_3m)

In [ ]:
# Analyze feature impact
print("\n📊 Retention Drivers Analysis:")

In [ ]:
# 1. First order value impact
first_value_retention = retention_features.groupby(
    pd.cut(retention_features['total_amount'], bins=[0, 200, 400, 600, 1000, 2000])
)['retained_3m'].mean()
print("\n• First Order Value Impact:")
for interval, rate in first_value_retention.items():
    print(f"  - {interval}: {rate*100:.1f}%")

In [ ]:
# 2. Channel impact
channel_retention = retention_features.groupby('acquisition_channel')['retained_3m'].mean().sort_values(ascending=False)
print("\n• Acquisition Channel Impact:")
for channel, rate in channel_retention.items():
    print(f"  - {channel}: {rate*100:.1f}%")

In [ ]:
# 3. Premium impact
premium_retention = retention_features.groupby('is_premium_member')['retained_3m'].mean()
print(f"\n• Premium Impact:")
print(f"  - Premium: {premium_retention.get(True, 0)*100:.1f}%")
print(f"  - Non-Premium: {premium_retention.get(False, 0)*100:.1f}%")

In [ ]:
# 4. Device impact
device_retention = retention_features.groupby('device_type')['retained_3m'].mean()
print("\n• Device Type Impact:")
for device, rate in device_retention.items():
    print(f"  - {device}: {rate*100:.1f}%")

In [ ]:
# 5. Age band impact
age_retention = retention_features.groupby('age_band')['retained_3m'].mean().sort_values(ascending=False)
print("\n• Age Band Impact:")
for age, rate in age_retention.items():
    print(f"  - {age}: {rate*100:.1f}%")

In [ ]:
# Visualize retention drivers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Key Drivers of 3-Month Retention', fontsize=16, fontweight='bold')

In [ ]:
# 5.1 First Order Value
ax = axes[0, 0]
first_value_retention.plot(kind='bar', ax=ax, color='#2ecc71', alpha=0.7)
ax.set_title('Retention by First Order Value')
ax.set_xlabel('First Order Value (₹)')
ax.set_ylabel('3-Month Retention Rate (%)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 5.2 Acquisition Channel
ax = axes[0, 1]
channel_retention.plot(kind='bar', ax=ax, color='#3498db', alpha=0.7)
ax.set_title('Retention by Acquisition Channel')
ax.set_xlabel('Channel')
ax.set_ylabel('3-Month Retention Rate (%)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 5.3 Device & Premium
ax = axes[1, 0]
device_retention.plot(kind='bar', ax=ax, color='#9b59b6', alpha=0.7)
ax.set_title('Retention by Device Type')
ax.set_xlabel('Device')
ax.set_ylabel('3-Month Retention Rate (%)')
ax.tick_params(axis='x', rotation=0)

In [ ]:
# 5.4 Age Band
ax = axes[1, 1]
age_retention.plot(kind='bar', ax=ax, color='#e67e22', alpha=0.7)
ax.set_title('Retention by Age Band')
ax.set_xlabel('Age Band')
ax.set_ylabel('3-Month Retention Rate (%)')
ax.tick_params(axis='x', rotation=0)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/retention_drivers.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
11. SUMMARY & KEY INSIGHTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("COHORT RETENTION ANALYSIS - KEY INSIGHTS")
print("="*80